In [8]:
#%% [markdown]
# # Q-Learning 逃離 21×11 迷宮（含 5 個寶藏 + Mask State + Early Stop  
# + 動態獎勵塑形 + 避免撞牆重選動作）

#%%  
import numpy as np
import random

# ─── 一、參數與地圖設定 ─────────────────────────────────────────  
WIDTH, HEIGHT = 21, 11  
NUM_POS = WIDTH * HEIGHT           # 231  

treasure_list = [  
    (0, 6), (3, 16), (8, 2), (10, 2), (10, 17)  
]  
treasures = set(treasure_list)  

walls_coords = [  
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),  
    (2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),  
    (3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),  
    (5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),  
    (7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),  
    (9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)  
]  
walls = set(walls_coords)  

START = (0,0)  
GOAL  = (20,10)  

# Q-learning 超參數  
MAX_EPISODES = 3000  
MAX_STEPS    = 1000  

EPSILON   = 0.9  
EPS_DECAY  = 0.999  
EPS_MIN    = 0.01  
ALPHA      = 0.5  
GAMMA      = 0.995  

# Early stop  
patience   = 500  
no_improve = 0  

ACTIONS     = ['up','down','left','right']  
NUM_ACTIONS = len(ACTIONS)  

NUM_MASK   = 1 << len(treasure_list)  # 32  
NUM_STATES = NUM_POS * NUM_MASK       # 7392  

Q = np.zeros((NUM_STATES, NUM_ACTIONS))  


# 坐標 ↔ 索引  
def to_index(pos):  
    return pos[1] * WIDTH + pos[0]  

def to_pos(idx):  
    return (idx % WIDTH, idx // WIDTH)  

def encode_state(pos, mask):  
    return to_index(pos) * NUM_MASK + mask  

def compute_target(pos, action):  
    x,y = pos  
    if action=='up':    y = max(0,   y-1)  
    elif action=='down':y = min(HEIGHT-1, y+1)  
    elif action=='left':x = max(0,   x-1)  
    else:               x = min(WIDTH-1,  x+1)  
    return (x,y)  

def nearest_target(pos, mask):  
    remaining = [t for i,t in enumerate(treasure_list) if not (mask & (1<<i))]  
    return min(remaining, key=lambda t: abs(t[0]-pos[0]) + abs(t[1]-pos[1])) if remaining else GOAL  

def manhattan(a,b):  
    return abs(a[0]-b[0]) + abs(a[1]-b[1])  


#%% [markdown]  
# ## 訓練迴圈（加入避免撞牆重選動作）  

#%%  
best_path  = None  
best_steps = MAX_STEPS + 1  
best_score = -1  
epsilon    = EPSILON  

for episode in range(1, MAX_EPISODES+1):  
    pos   = START  
    mask  = 0  
    state = encode_state(pos, mask)  
    path  = [state]  
    score = 0  

    for step in range(1, MAX_STEPS+1):  
        # 先選一個合法動作：如果撞牆或留地則重選  
        while True:  
            if random.random() < epsilon:  
                a = random.randrange(NUM_ACTIONS)  
            else:  
                a = np.argmax(Q[state])  
            action = ACTIONS[a]  
            attempted = compute_target(pos, action)  
            if attempted in walls or attempted == pos:  
                continue  
            break  

        new_pos, new_mask = attempted, mask  

        # 基本獎懲  
        if new_pos in treasures:  
            i = treasure_list.index(new_pos)  
            if not (mask & (1<<i)):  
                new_mask |= (1<<i)  
                r = +10  
                score += 1  
            else:  
                r = -1  
        elif new_pos == GOAL:  
            r = +58 if new_mask == (NUM_MASK-1) else -50  
        else:  
            r = -1  

        # 動態獎勵塑形：朝最近未收寶藏（或終點）移動則 +2  
        target = nearest_target(pos, mask)
        dist0 = manhattan(pos, target)
        dist1 = manhattan(new_pos, target)
        if dist1 < dist0:
            r += +2
        elif dist1 > dist0:
            r += -2   # 小额惩罚，别超过主 reward

        new_state = encode_state(new_pos, new_mask)  

        # Q 更新  
        q_pred = Q[state, a]  
        q_tgt  = r if (new_pos==GOAL and new_mask==(NUM_MASK-1)) else r + GAMMA * np.max(Q[new_state])  
        Q[state, a] += ALPHA * (q_tgt - q_pred)  

        pos, mask, state = new_pos, new_mask, new_state  
        path.append(state)  

        if new_pos == GOAL and new_mask == (NUM_MASK-1):  
            break  

    # ε 衰減  
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)  

    # 記錄最佳  
    improved = False  
    if pos==GOAL and mask==(NUM_MASK-1) and step<best_steps:  
        best_steps, best_score, best_path = step, score, path.copy()  
        improved = True  

    # Early stop  
    no_improve = 0 if improved else no_improve + 1  
    if epsilon<=EPS_MIN and no_improve>=patience:  
        print(f"Early stop at episode {episode}, no improvement for {no_improve}.")  
        break  

    if episode % 100 == 0:  
        print(f"Ep{episode:4d} | ε={epsilon:.3f} | best_steps={best_steps}")  


#%% [markdown]  
# ## 輸出 & 儲存結果  

#%%  
if best_path is None:  
    print("未找到收集 5 寶並到終點的路徑")  
else:  
    print("=== 最佳結果 ===")  
    print(f"步數：{best_steps}，寶藏：{best_score}（應為 {len(treasure_list)}）")  
    np.save('q_table.npy', Q)  
    # 文字化路徑顯示  
    maze = [[' ']*WIDTH for _ in range(HEIGHT)]  
    for x,y in walls_coords:  maze[y][x]='X'  
    for x,y in treasure_list: maze[y][x]='O'  
    maze[START[1]][START[0]]='S'; maze[GOAL[1]][GOAL[0]]='G'  
    for s in best_path:  
        x,y = to_pos(s//NUM_MASK)  
        if maze[y][x]==' ': maze[y][x]='.'  
    print("\n路徑（. 為走過的路徑）：")  
    for row in maze:  
        print(''.join(row))  

# EOF

Ep 100 | ε=0.814 | best_steps=1001
Ep 200 | ε=0.737 | best_steps=1001
Ep 300 | ε=0.667 | best_steps=1001
Ep 400 | ε=0.603 | best_steps=1001
Ep 500 | ε=0.546 | best_steps=1001
Ep 600 | ε=0.494 | best_steps=1001
Ep 700 | ε=0.447 | best_steps=1001
Ep 800 | ε=0.404 | best_steps=1001
Ep 900 | ε=0.366 | best_steps=1001
Ep1000 | ε=0.331 | best_steps=1001
Ep1100 | ε=0.299 | best_steps=1001
Ep1200 | ε=0.271 | best_steps=1001
Ep1300 | ε=0.245 | best_steps=1001
Ep1400 | ε=0.222 | best_steps=1001
Ep1500 | ε=0.201 | best_steps=1001
Ep1600 | ε=0.182 | best_steps=1001
Ep1700 | ε=0.164 | best_steps=1001
Ep1800 | ε=0.149 | best_steps=1001
Ep1900 | ε=0.134 | best_steps=1001
Ep2000 | ε=0.122 | best_steps=1001
Ep2100 | ε=0.110 | best_steps=1001
Ep2200 | ε=0.100 | best_steps=1001
Ep2300 | ε=0.090 | best_steps=1001
Ep2400 | ε=0.082 | best_steps=1001
Ep2500 | ε=0.074 | best_steps=1001
Ep2600 | ε=0.067 | best_steps=1001
Ep2700 | ε=0.060 | best_steps=1001
Ep2800 | ε=0.055 | best_steps=1001
Ep2900 | ε=0.049 | b